# 2 — Data Preprocessing Pipeline

**Project:** Predictive Modeling of US Used Vehicle Prices  
**Course:** ENGR422 — Applied Machine Learning  
**Authors:** Eren Acar Başaran, Ahmet Aybars Pektaş  

---

This notebook covers **Work Package 2 — Preprocessing Pipeline** from the project proposal.

**Deliverables:**
- D2.1: Cleaned and Split Dataset ready for modeling
- D2.2: Scikit-Learn Preprocessing Pipeline Script

## 2.1 — Imports & Load Raw Data

Import libraries (Pandas, NumPy, Scikit-Learn). Load the raw dataset from `../data/vehicles.csv` and print the shape. No filtering here — that happens in section 2.3.

In [1]:
# 2.1 — Imports & Load Raw Data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
DATA_PATH = "../data/vehicles.csv"

# Load raw dataset
df = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (426880, 26)


## 2.2 — Feature Selection & Column Dropping

Based on the EDA findings, drop columns that are not useful for modeling:
- **Identifiers:** `id`, `url`, `region_url`, `image_url`, `VIN` — not predictive
- **Free-text:** `description` — NLP out of scope
- **Geography:** `lat`, `long`, `region` — `state` kept instead
- **Excessive missing:** `county` (100%), `size` (71.8%), `cylinders` (41.6%) — too many NaN to be useful
- **Temporal:** `posting_date` — not relevant to price

Print the remaining columns and shape after dropping.

In [2]:
# 2.2 — Feature Selection & Column Dropping

# Identifiers — no predictive value
# Free-text / URL fields — not used (text analysis is out of scope)
# Columns with excessive missing data or no data at all (county is 100% NaN)
# Location coordinates — too granular, state is kept instead
# posting_date — not a vehicle feature
# region — redundant with state
# size — over 70% missing in EDA
cols_to_drop = [
    "id", "url", "region_url", "image_url",  # identifiers / URLs
    "description",                             # free-text, out of scope
    "VIN",                                     # identifier
    "county",                                  # 100% NaN
    "lat", "long",                             # coordinates, too granular
    "posting_date",                            # not a vehicle feature
    "region",                                  # redundant with state
    "size",                                    # >70% missing
    "cylinders",                               # 41.6% missing, correlated with manufacturer/model
]

df = df.drop(columns=cols_to_drop, errors="ignore")

# state kept for potential use in future iterations
print(f"Shape after dropping columns: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Shape after dropping columns: (426880, 13)
Remaining columns: ['price', 'year', 'manufacturer', 'model', 'condition', 'fuel', 'odometer', 'title_status', 'transmission', 'drive', 'type', 'paint_color', 'state']


## 2.3 — Outlier Filtering

Apply hard-bound filtering based on EDA findings:
- Drop rows where `price` is NaN.
- Keep rows where `price` >= $500 **and** `price` <= $100,000 (removes spam listings and unrealistic values).
- Drop rows where `odometer` > 300,000 miles.

Report the number of rows before and after filtering.

In [3]:
# 2.3 — Outlier Filtering
rows_before = len(df)

# Drop rows where price is NaN
df = df.dropna(subset=["price"])



# Price filter: hard min $500, hard max $100,000
# Keeps legitimate vehicles while removing $0/$1 spam and unrealistic listings

PRICE_MIN = 500
PRICE_MAX = 100_000

df = df[(df["price"] >= PRICE_MIN) & (df["price"] <= PRICE_MAX)]

# Odometer: drop rows above 300,000 miles

ODO_CAP = 300_000
df = df[df["odometer"].fillna(0) <= ODO_CAP]

rows_after = len(df)
print(f"Before filtering: {rows_before:,} rows")
print(f"After filtering:  {rows_after:,} rows")
print(f"Rows removed:     {rows_before - rows_after:,}")

Before filtering: 426,880 rows
After filtering:  381,427 rows
Rows removed:     45,453


## 2.4 — Train-Test Split

**!! CRITICAL: Split BEFORE any imputation or encoding to prevent data leakage !!**

- Separate features (`X`) and target (`y = price`).
- Perform an 80/20 train-test split using `train_test_split` with a fixed `random_state` for reproducibility.
- Report the shape of train and test sets.

In [4]:
# 2.4 — Train-Test Split
# !! CRITICAL: Split BEFORE any imputation or encoding to prevent data leakage !!

X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (305141, 12)
X_test:  (76286, 12)
y_train: (305141,)
y_test:  (76286,)


## 2.5 — Missing Value Imputation

Build imputation strategies (fit strictly on training data only, transform both train and test):

- **`year`** (0.3% missing): Group by odometer in 3,000-mile bins → median year of that bin. Fallback: overall median year.
- **`odometer`** (1% missing): Group by year → median odometer of that year. Fallback: overall median odometer.
- **`condition`** (40% missing): Odometer bins (0–50K, 50K–100K, 100K–150K, 150K–200K, 200K+) → most frequent condition in that bin. Fallback: `'good'`.
- **`manufacturer`** (4% missing): Group by model → most frequent manufacturer for that model. Fallback: `'unknown'`.
- **`model`** (1.2% missing): Group by manufacturer → most frequent model for that manufacturer. Fallback: `'unknown'`.
- **`fuel`, `transmission`, `drive`, `type`, `title_status`, `paint_color`, `state`**: 3-level cascade → group by model → group by manufacturer → global mode. Fallback: `'unknown'`.



In [5]:
# 2.5 — Missing Value Imputation
# In order to prevent data leakage, preprocessing pipelines will be used
# In this section, custom sklearn classes following the imputing strategies described above are defined

from sklearn.base import BaseEstimator, TransformerMixin

BIN_SIZE = 3000

class OdometerBinnedYearImputer(BaseEstimator, TransformerMixin):

    def __init__(self, bin_size=BIN_SIZE):
        self.bin_size = bin_size

    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=["year", "odometer"])
        odo_bin = (X["odometer"] // self.bin_size) * self.bin_size
        self.mapping_ = X.groupby(odo_bin)["year"].median().to_dict()
        self.fallback_ = X["year"].median()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        mask = X["year"].isna()
        odo_bin = (X["odometer"] // self.bin_size) * self.bin_size
        X.loc[mask, "year"] = odo_bin[mask].map(self.mapping_).fillna(self.fallback_)
        return X


class YearBinnedOdometerImputer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=["year", "odometer"])
        self.mapping_ = X.groupby("year")["odometer"].median().to_dict()
        self.fallback_ = X["odometer"].median()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        mask = X["odometer"].isna()
        X.loc[mask, "odometer"] = X.loc[mask, "year"].map(self.mapping_).fillna(self.fallback_)
        return X


class OdometerBinnedConditionImputer(BaseEstimator, TransformerMixin):

    def __init__(self, bin_edges=None, bin_labels=None, fallback="good"):
        self.bin_edges  = bin_edges  or [0, 50_000, 100_000, 150_000, 200_000, np.inf]
        self.bin_labels = bin_labels or ["0-50K", "50K-100K", "100K-150K", "150K-200K", "200K+"]
        self.fallback   = fallback

    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=["odometer", "condition"]).copy()
        X["_cond_bin"] = pd.cut(X["odometer"], bins=self.bin_edges, labels=self.bin_labels, right=False)
        valid = X.dropna(subset=["condition", "_cond_bin"])
        self.mapping_ = (
            valid.groupby(["_cond_bin", "condition"], observed=True)
            .size()
            .reset_index(name="_n")
            .sort_values("_n", ascending=False)
            .drop_duplicates("_cond_bin")
            .set_index("_cond_bin")["condition"]
            .to_dict()
        )
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        mask = X["condition"].isna()
        cond_bin = pd.cut(X["odometer"], bins=self.bin_edges, labels=self.bin_labels, right=False)
        X.loc[mask, "condition"] = cond_bin[mask].map(self.mapping_).fillna(self.fallback)
        return X


class ModelBinnedManufacturerImputer(BaseEstimator, TransformerMixin):

    def __init__(self, fallback="unknown"):
        self.fallback = fallback

    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=["model", "manufacturer"])
        valid = X.dropna(subset=["model", "manufacturer"])
        self.mapping_ = (
            valid.groupby(["model", "manufacturer"])
            .size()
            .reset_index(name="_n")
            .sort_values("_n", ascending=False)
            .drop_duplicates("model")
            .set_index("model")["manufacturer"]
            .to_dict()
        )
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        mask = X["manufacturer"].isna()
        X.loc[mask, "manufacturer"] = (
            X.loc[mask, "model"].map(self.mapping_).fillna(self.fallback)
        )
        return X


class CascadingCategoricalImputer(BaseEstimator, TransformerMixin):

    def __init__(self, target_cols=None, fallback="unknown"):
        self.target_cols = target_cols or ["fuel", "transmission", "drive", "type",
                                           "title_status", "paint_color", "state"]
        self.fallback = fallback

    def _vectorized_mode(self, df, group_col, value_col):
        valid = df.dropna(subset=[group_col, value_col])
        return (
            valid.groupby([group_col, value_col])
            .size()
            .reset_index(name="_n")
            .sort_values("_n", ascending=False)
            .drop_duplicates(group_col)
            .set_index(group_col)[value_col]
            .to_dict()
        )

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.mode_by_model_ = {}
        self.mode_by_mfr_   = {}
        self.mode_overall_  = {}

        for col in self.target_cols:
            self.mode_by_model_[col] = self._vectorized_mode(X, "model", col)
            self.mode_by_mfr_[col]   = self._vectorized_mode(X, "manufacturer", col)
            mode = X[col].mode()
            self.mode_overall_[col]  = mode.iloc[0] if not mode.empty else self.fallback

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        for col in self.target_cols:
            mask = X[col].isna()
            if not mask.any():
                continue

            imputed = pd.Series(self.mode_overall_[col], index=X.index)

            mfr_match = X["manufacturer"].map(self.mode_by_mfr_[col])
            imputed = imputed.where(mfr_match.isna(), mfr_match)

            model_match = X["model"].map(self.mode_by_model_[col])
            imputed = imputed.where(model_match.isna(), model_match)

            X.loc[mask, col] = imputed[mask]

        return X




## 2.6 — Preprocessing ColumnTransformer

Assemble two full preprocessing pipelines using the custom imputers from §2.5:

**Stage 1 — Sequential imputation** (shared structure, fresh instances per pipeline):  
A `Pipeline` of all custom imputers operating on the full DataFrame. Must be sequential — year imputation depends on odometer and vice versa, so parallel `ColumnTransformer` branches are not viable here.

**Stage 2 — Encoding `ColumnTransformer`** (parallel branches):
| Branch | Columns | Transformer |
|---|---|---|
| `num` | `year`, `odometer` | `StandardScaler` *(linear)* / `passthrough` *(trees)* |
| `ord` | `condition` | `OrdinalEncoder` with explicit order: `salvage→fair→good→excellent→like new→new` |
| `ohe` | `fuel`, `transmission`, `drive`, `type`, `title_status` | `OneHotEncoder(drop='first')` |
| `target` | `manufacturer`, `model`, `paint_color`, `state` | `TargetEncoder` — learns mean price per category from `y_train` during `fit` |

Two preprocessors are produced: `preprocessor_linear` (numerics scaled) and `preprocessor_trees` (numerics passed through). Both are ready to be composed into model pipelines in §2.7.

In [10]:
# 2.6 — Preprocessing ColumnTransformer

# ============================================================
# Column group definitions
# ============================================================
NUM_YEAR       = ["year"]
NUM_ODO        = ["odometer"]
ORD_COL        = ["condition"]
LOW_CARD_COLS  = ["fuel", "transmission", "drive", "type", "title_status"]
HIGH_CARD_COLS = ["manufacturer", "model", "paint_color", "state"]

# salvage=0 … new=5; unknown_value=-1 avoids collision with those indices
CONDITION_CATEGORIES = [["salvage", "fair", "good", "excellent", "like new", "new"]]

# ================================================================
# Custom Target Encoder: Built-in Target Encoder of sklearn
# causes hundred thousands of lambda applications, causing 
# high computaiton times, so we built a custom tartget encoder
# ================================================================

class MeanTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Replaces each category with the mean target value learned from training data.
    Operates on all columns passed to it by ColumnTransformer.
    Unseen categories fall back to the global mean.
    Used instead of sklearn's TargetEncoder which allocates a dense
    (n_samples x n_categories) indicator matrix — infeasible for high-cardinality
    columns like 'model' with 24K+ unique values.
    """

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y_arr = np.asarray(y, dtype=float)
        self.global_mean_ = float(y_arr.mean())
        self.encodings_ = {}
        for col in X.columns:
            tmp = pd.DataFrame({"cat": X[col].values, "target": y_arr})
            self.encodings_[col] = tmp.groupby("cat")["target"].mean().to_dict()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col in X.columns:
            X[col] = X[col].map(self.encodings_.get(col, {})).fillna(self.global_mean_)
        return X
    
    
# ============================================================
# Stage 1: Sequential imputation pipeline (full DataFrame)
# Must be sequential — year imputation uses odometer and vice versa
# ============================================================
def _make_imputation_pipeline():
    return Pipeline([
        ("year_imp", OdometerBinnedYearImputer()),
        ("odo_imp",  YearBinnedOdometerImputer()),
        ("cond_imp", OdometerBinnedConditionImputer()),
        ("mfr_imp",  ModelBinnedManufacturerImputer()),
        ("cat_imp",  CascadingCategoricalImputer()),
    ])

# ============================================================
# Odometer: log1p transform then StandardScaler
# log1p compresses the right tail; StandardScaler centers the result
# ============================================================
_odo_scaler = Pipeline([
    ("log1p",  FunctionTransformer(np.log1p, inverse_func=np.expm1)),
    ("scaler", StandardScaler()),
])

# ============================================================
# Stage 2: Encoding — linear regression variant (scaled numerics)
# ============================================================
encoding_linear = ColumnTransformer([
    ("year",   StandardScaler(),  NUM_YEAR),
    ("odo",    _odo_scaler,       NUM_ODO),
    ("ord",    OrdinalEncoder(categories=CONDITION_CATEGORIES,
                              handle_unknown="use_encoded_value",
                              unknown_value=-1),
               ORD_COL),
    ("ohe",    OneHotEncoder(drop="first",
                             sparse_output=False,
                             handle_unknown="ignore"),
               LOW_CARD_COLS),
    ("target", MeanTargetEncoder(),
               HIGH_CARD_COLS),
], remainder="drop")

# ============================================================
# Stage 2: Encoding — tree model variant (no scaling)
# ============================================================
encoding_trees = ColumnTransformer([
    ("year",   "passthrough",     NUM_YEAR),
    ("odo",    "passthrough",     NUM_ODO),
    ("ord",    OrdinalEncoder(categories=CONDITION_CATEGORIES,
                              handle_unknown="use_encoded_value",
                              unknown_value=-1),
               ORD_COL),
    ("ohe",    OneHotEncoder(drop="first",
                             sparse_output=False,
                             handle_unknown="ignore"),
               LOW_CARD_COLS),
    ("target", MeanTargetEncoder(),
               HIGH_CARD_COLS),
], remainder="drop")



## 2.7 — Pipeline Assembly

*Preprocessing pipelines for linear and tree models are assembled. The cleaned DataFrames are saved as CSV in section 2.8 and loaded directly by the model notebooks.*

In [16]:
# ============================================================
# Full preprocessing pipelines
# ============================================================
preprocessor_linear = Pipeline([
    ("impute", _make_imputation_pipeline()),
    ("encode", encoding_linear),
])

preprocessor_tree = Pipeline([
    ("impute", _make_imputation_pipeline()),
    ("encode", encoding_trees),
])

print("preprocessor_linear steps:")
for name, step in preprocessor_linear.steps:
    print(f"  {name}: {step}")

print("\npreprocessor_tree steps:")
for name, step in preprocessor_tree.steps:
    print(f"  {name}: {step}")
preprocessor_tree

preprocessor_linear steps:
  impute: Pipeline(steps=[('year_imp', OdometerBinnedYearImputer()),
                ('odo_imp', YearBinnedOdometerImputer()),
                ('cond_imp',
                 OdometerBinnedConditionImputer(bin_edges=[0, 50000, 100000,
                                                           150000, 200000,
                                                           inf],
                                                bin_labels=['0-50K', '50K-100K',
                                                            '100K-150K',
                                                            '150K-200K',
                                                            '200K+'])),
                ('mfr_imp', ModelBinnedManufacturerImputer()),
                ('cat_imp',
                 CascadingCategoricalImputer(target_cols=['fuel',
                                                          'transmission',
                                                          'drive', 'ty

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute', ...), ('encode', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('year_imp', ...), ('odo_imp', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,bin_size,3000
,bin_edges,"[0, 50000, ...]"
,bin_labels,"['0-50K', '50K-100K', ...]"


## 2.8 — Verification & Sanity Checks

Verify the final preprocessed dataset before saving:

- **Shape check:** confirm `X_train` and `X_test` have identical columns (32 features after encoding)
- **Zero missing values:** confirm no NaN remains in either split
- **Column inspection:** print all 32 feature names to verify encoding worked correctly
- **Sample rows:** display first 3 rows of `X_train` to visually confirm encoded values
- **Save to disk:** export `X_train`, `X_test`, `y_train`, `y_test` as CSV files to `../data/` for use by model notebooks

No transformations happen here — this is inspection and export only.

In [18]:
# 2.8 — Verification & Sanity Checks

# Shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")
print()

# All column names after encoding
print(f"Columns ({len(X_train.columns)}):")
print(X_train.columns.tolist())
print()

print("Applying scaling, imputing and encoding to train and test data for linear regression model...")
X_train_linear_processed = preprocessor_linear.fit_transform(X_train, y_train)
X_test_linear_processed = preprocessor_linear.transform(X_test)
print("Success!\n")

# Zero missing values confirmation encoding confirmation
print("Missing values in X_train:", np.isnan(X_train_linear_processed).sum())
print("Missing values in X_test: ", np.isnan(X_test_linear_processed).sum())
print()
print(f"Shape of the processed train data: {X_train_linear_processed.shape}")
print(f"Shape of the processed test data: {X_test_linear_processed.shape}")


#For tree models preprocessor
print("Applying scaling, imputing and encoding to train and test data for tree models...")
X_train_tree_processed = preprocessor_tree.fit_transform(X_train, y_train)
X_test_tree_processed = preprocessor_tree.transform(X_test)
print("Success!\n")

# Zero missing values confirmation encoding confirmation
print("Missing values in X_train:", np.isnan(X_train_tree_processed).sum())
print("Missing values in X_test: ", np.isnan(X_test_tree_processed).sum())
print()
print(f"Shape of the processed train data: {X_train_tree_processed.shape}")
print(f"Shape of the processed test data: {X_test_tree_processed.shape}")


# Save to CSV
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)
pd.DataFrame(X_train_linear_processed).to_csv("../data/X_train_linear_processed.csv", index=False)
pd.DataFrame(X_test_linear_processed).to_csv("../data/X_test_linear_processed.csv", index=False)
pd.DataFrame(X_train_tree_processed).to_csv("../data/X_train_tree_processed.csv", index=False)
pd.DataFrame(X_test_tree_processed).to_csv("../data/X_test_tree_processed.csv", index=False)

print(f"\nSaved to ../data/:")
print(f"  X_train.csv  ({X_train.shape})")
print(f"  X_test.csv   ({X_test.shape})")
print(f"  y_train.csv  ({y_train.shape})")
print(f"  y_test.csv   ({y_test.shape})")
print(f"  X_train_linear_processed.csv  ({X_train_linear_processed.shape})")
print(f"  X_test_linear_processed.csv   ({X_test_linear_processed.shape})")

X_train shape: (305141, 12)
X_test shape:  (76286, 12)
y_train shape: (305141,)
y_test shape:  (76286,)

Columns (12):
['year', 'manufacturer', 'model', 'condition', 'fuel', 'odometer', 'title_status', 'transmission', 'drive', 'type', 'paint_color', 'state']

Applying scaling, imputing and encoding to train and test data for linear regression model...
Success!

Missing values in X_train: 0
Missing values in X_test:  0

Shape of the processed train data: (305141, 32)
Shape of the processed test data: (76286, 32)
Applying scaling, imputing and encoding to train and test data for tree models...
Success!

Missing values in X_train: 0
Missing values in X_test:  0

Shape of the processed train data: (305141, 32)
Shape of the processed test data: (76286, 32)

Saved to ../data/:
  X_train.csv  ((305141, 12))
  X_test.csv   ((76286, 12))
  y_train.csv  ((305141,))
  y_test.csv   ((76286,))
  X_train_linear_processed.csv  ((305141, 32))
  X_test_linear_processed.csv   ((76286, 32))
